# 01 — Kiểm tra và làm sạch sự kiện RetailRocket

Thực hiện bước B–D: **kiểm tra schema → bỏ trùng tuyệt đối → chuẩn hóa thời gian UTC → lưu và đọc kiểm tra Parquet**.

- Đầu vào: `data/raw/events.csv`, giữ nguyên.
- Đầu ra: `data/processed/events_clean.parquet` và `reports/events_clean/metrics.json`.
- Chạy lại sẽ cập nhật các đầu ra này; notebook 02 cần chạy sau notebook 01 để dùng đúng phiên bản dữ liệu.
- Notebook tự chứa các hàm hỗ trợ, không cần file `.py` hoặc báo cáo raw audit.

Trong VS Code/Jupyter, chọn kernel Python của môi trường ảo `.venv` trong project (tạo bằng `python -m venv .venv` rồi `pip install -r requirements.txt`, xem README mục *Chạy nhanh*). Dùng **Restart Kernel and Run All** để chạy từ trạng thái sạch, hoặc **Shift+Enter** lần lượt từ trên xuống.

Lỗi thiếu `ipykernel` phải xử lý ở môi trường Python vì kernel chưa khởi động để chạy cell cài đặt. Cell đầu in phiên bản Python/pandas/pyarrow thực tế để kiểm tra.

Sau khi hoàn thành, tiếp tục [notebook 02](02_tao_phien_nhom_timestamp_va_chia_tap.ipynb). Xem [hướng dẫn tiền xử lý](../docs/HUONG_DAN_TIEN_XU_LY_DU_LIEU.md) cho toàn bộ lộ trình.

## 1. Đường dẫn, quy tắc và hàm hỗ trợ

Tìm project từ thư mục làm việc hiện tại hoặc thư mục cha. Mở notebook trong project hoặc `notebooks/`.

- `check_columns()`: kiểm tra tên và thứ tự năm cột gốc.
- `invalid_integer()`: kiểm tra chữ số nguyên không âm và giới hạn `int64`; trường optional được thiếu.
- `fingerprint()`: tính dung lượng/SHA-256 theo byte của tệp (truy vết đúng tệp).
- `content_sha256()`: SHA-256 theo nội dung bảng, sắp theo khóa chính; dùng để đối chiếu giữa các máy vì mã băm byte của Parquet đổi theo phiên bản pyarrow.
- `require()`: dừng với thông báo rõ khi điều kiện không đạt.
- `write_json()`: ghi UTF-8 với xuống dòng LF (`newline="\n"`) qua tệp tạm rồi thay thế báo cáo chính, để mã băm JSON giống nhau trên Windows/macOS/Linux.

Khoảng ngày kỳ vọng là cấu hình riêng của bộ RetailRocket này. Giá trị ngoài khoảng sẽ được báo để xem lại nguồn/đơn vị, không bị tự xóa.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
import pyarrow as pa
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next(
    (path for path in [cwd, *cwd.parents]
     if (path / "data/raw/events.csv").is_file()), None
)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ project hoặc notebooks/ có data/raw/events.csv.")

SOURCE = ROOT / "data/raw/events.csv"
OUTPUT = ROOT / "data/processed/events_clean.parquet"
REPORT_DIR = ROOT / "reports/events_clean"
REPORT_PATH = REPORT_DIR / "metrics.json"
COLUMNS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
REQUIRED = ["timestamp", "visitorid", "event", "itemid"]
ALLOWED_EVENTS = ["view", "addtocart", "transaction"]
EXPECTED_START = pd.Timestamp("2015-05-03", tz="UTC")
EXPECTED_END_EXCLUSIVE = pd.Timestamp("2015-09-19", tz="UTC")
RUN_ID = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")


def check_columns(frame, expected):
    if list(frame.columns) != expected:
        raise ValueError(f"Tên/thứ tự cột không đúng: {list(frame.columns)}; cần: {expected}")


def invalid_integer(series, optional=False):
    valid_format = series.str.fullmatch(r"[0-9]+", na=False).astype(bool)
    # So sánh chuỗi để tránh làm tròn qua float khi số vượt giới hạn int64.
    significant = series.str.lstrip("0").fillna("")
    length = significant.str.len()
    in_range = (
        length.lt(19).astype(bool)
        | (length.eq(19).astype(bool) & significant.le("9223372036854775807").astype(bool))
    )
    invalid = ~(valid_format & in_range)
    return (invalid & series.notna()) if optional else invalid

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def fingerprint(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return {"bytes": path.stat().st_size, "sha256": digest.hexdigest()}


def content_sha256(frame, key):
    """SHA-256 theo nội dung bảng: sắp theo khóa chính rồi băm từng dòng.

    Không phụ thuộc cách pyarrow/pandas xếp byte trong tệp Parquet, nên dùng để
    đối chiếu giữa các máy. `fingerprint()` vẫn giữ mã băm theo byte để truy vết tệp.
    """
    ordered = frame.sort_values(key, kind="stable").reset_index(drop=True)
    digest = hashlib.sha256(json.dumps([str(c) for c in ordered.columns]).encode("utf-8"))
    row_hashes = pd.util.hash_pandas_object(ordered, index=False).to_numpy(dtype="<u8")
    digest.update(row_hashes.tobytes())
    return digest.hexdigest()


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    try:
        temporary.write_text(
            json.dumps(value, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8", newline="\n",
        )
        json.loads(temporary.read_text(encoding="utf-8"))
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


# Kiểm tra trường hợp biên của ID, không dùng dữ liệu raw để làm ví dụ giả.
integer_examples = pd.Series(
    ["0", "0012", "9223372036854775807", "9223372036854775808", "-1", "1.5", "abc", "", pd.NA],
    dtype="string",
)
assert invalid_integer(integer_examples).tolist() == [False, False, False, True, True, True, True, True, True]
assert not bool(invalid_integer(integer_examples, optional=True).iloc[-1])
_content_example = pd.DataFrame({"event_id": [3, 1, 2], "value": ["c", "a", "b"]})
assert content_sha256(_content_example, ["event_id"]) == content_sha256(_content_example.iloc[::-1], ["event_id"])
assert content_sha256(_content_example, ["event_id"]) != content_sha256(_content_example.assign(value="x"), ["event_id"])
del _content_example

print("Phiên bản:", sys.version.split()[0], "| pandas:", pd.__version__, "| pyarrow:", pa.__version__)
print("Project:", ROOT.name)
print("Đầu vào:", SOURCE.relative_to(ROOT).as_posix())

## 2. Đọc dữ liệu và quan sát ban đầu

Đọc dạng chuỗi trước để kiểm tra biểu diễn gốc. Schema gồm tên cột, kiểu dữ liệu, trường được phép thiếu và điều kiện hợp lệ; ở bước này chưa ép kiểu số.

`fingerprint()` ghi dung lượng và SHA-256 để nhận diện tệp đầu vào, sau đó kiểm tra raw không thay đổi trong lúc xử lý.

In [ ]:
source_before = fingerprint(SOURCE)
previous_report = (
    json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    if REPORT_PATH.is_file() else None
)
raw = pd.read_csv(SOURCE, dtype="string")
check_columns(raw, COLUMNS)
require(len(raw) > 0, "events.csv không có sự kiện để xử lý.")

print(f"Dòng raw: {len(raw):,} | Số cột: {len(raw.columns)}")
display(raw.head())
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum()}))
display(raw["event"].value_counts(dropna=False).rename("raw_events"))

## 3. Kiểm tra schema và quan hệ giữa các cột

- Bốn trường `timestamp`, `visitorid`, `event`, `itemid` bắt buộc có.
- ID/timestamp phải là chuỗi chữ số nguyên không âm, không vượt 9.223.372.036.854.775.807 (giới hạn int64).
- Chỉ chấp nhận `view`, `addtocart`, `transaction`.
- `transactionid` được thiếu với view/addtocart, bắt buộc có khi mua.
- Nếu có mã giao dịch ở hành vi không phải mua, dừng để kiểm tra nguồn.

Không dùng `dropna()` trên toàn bộ bảng: cách đó sẽ xóa các lượt xem/thêm giỏ hợp lệ. Một dòng có thể vi phạm nhiều điều kiện, nên cần đếm riêng tổng dòng lỗi.

Nếu có lỗi, lưu chỉ số dòng nguồn và lý do vào CSV riêng theo lần chạy rồi dừng. Các điều kiện lỗi được chuyển về boolean đầy đủ để vẫn ghi được lý do khi có giá trị thiếu.

In [ ]:
flags = pd.DataFrame(index=raw.index)
flags["missing_required"] = raw[REQUIRED].isna().any(axis=1)
for column in ["timestamp", "visitorid", "itemid", "transactionid"]:
    flags[f"invalid_{column}"] = invalid_integer(
        raw[column], optional=(column == "transactionid")
    )
flags["invalid_event"] = ~raw["event"].isin(ALLOWED_EVENTS)
flags["purchase_missing_transactionid"] = (
    raw["event"].eq("transaction") & raw["transactionid"].isna()
)
flags["nonpurchase_has_transactionid"] = (
    ~raw["event"].eq("transaction") & raw["transactionid"].notna()
)
flags = flags.fillna(False).astype(bool)
bad_rows = flags.any(axis=1)

display(flags.sum().rename("violation_count"))
print("Tổng dòng có ít nhất một lỗi:", int(bad_rows.sum()))
if bad_rows.any():
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    errors = raw.loc[bad_rows].copy()
    errors.insert(0, "source_row_index", errors.index)
    errors["reason"] = flags.loc[bad_rows].apply(
        lambda row: ";".join(row.index[row]), axis=1
    )
    error_path = REPORT_DIR / f"schema_errors_{RUN_ID}.csv"
    errors.to_csv(error_path, index=False, encoding="utf-8-sig")
    display(errors.head(10))
    raise ValueError(f"Có lỗi schema; đã lưu dòng nguồn và lý do tại {error_path.relative_to(ROOT).as_posix()}")

## 4. Bỏ trùng tuyệt đối và giữ dấu vết dòng nguồn

Chỉ bỏ dòng trùng cả **năm cột gốc**. Xem lại cùng item ở thời điểm khác vẫn được giữ. Không bỏ trùng theo riêng visitor–item hoặc transactionid.

`event_id` là chỉ số dòng gốc (bắt đầu từ 0, không tính header), có thể không liên tiếp sau bỏ trùng. Không đưa cột này vào khóa phát hiện trùng.

In [ ]:
duplicate_mask = raw.duplicated(subset=COLUMNS, keep="first")
duplicate_count = int(duplicate_mask.sum())
duplicates_by_event = {
    str(key): int(value)
    for key, value in raw.loc[duplicate_mask, "event"].value_counts().items()
}
events = raw.loc[~duplicate_mask].copy()
events["event_id"] = events.index.astype("int64")

print(f"Dòng trước: {len(raw):,}")
print(f"Dòng trùng bị loại: {duplicate_count:,}")
print(f"Dòng giữ lại: {len(events):,}")
print("Trùng theo hành vi:", duplicates_by_event)
display(raw.loc[duplicate_mask].head())
assert len(raw) == len(events) + duplicate_count
assert not events.duplicated(subset=COLUMNS).any()

## 5. Chuẩn hóa kiểu và thời gian UTC

Timestamp RetailRocket có đơn vị **mili-giây từ Unix epoch**. Giữ timestamp gốc và thêm `event_time` có timezone UTC; không làm tròn về giây/phút và không cộng 7 giờ vào dữ liệu.

`errors="coerce"` biến thời gian không chuyển được thành `NaT` để kiểm tra. Notebook còn đối chiếu khoảng ngày đã được xác nhận trong báo cáo: 03/05–18/09/2015 UTC. Nếu ngoài khoảng, dừng kiểm tra nguồn/đơn vị thay vì tự xóa dòng.

In [ ]:
for column in ["timestamp", "visitorid", "itemid"]:
    events[column] = events[column].astype("int64")
events["transactionid"] = events["transactionid"].astype("Int64")
events["event"] = pd.Categorical(events["event"], categories=ALLOWED_EVENTS)
events["event_time"] = pd.to_datetime(
    events["timestamp"], unit="ms", origin="unix", utc=True, errors="coerce"
)

invalid_time = events["event_time"].isna()
if invalid_time.any():
    display(events.loc[invalid_time].head(10))
    raise ValueError("Có timestamp không chuyển được sang datetime.")

expected_start = EXPECTED_START
expected_end_exclusive = EXPECTED_END_EXCLUSIVE
outside_expected_range = (
    events["event_time"].lt(expected_start)
    | events["event_time"].ge(expected_end_exclusive)
)
if outside_expected_range.any():
    display(events.loc[outside_expected_range].head(10))
    raise ValueError("Thời gian ngoài khoảng RetailRocket đã xác nhận; kiểm tra nguồn/đơn vị.")

print("Bắt đầu:", events["event_time"].min())
print("Kết thúc:", events["event_time"].max())
display(events[["timestamp", "event_time"]].head())

## 6. Sắp xếp và kiểm tra kết quả trong bộ nhớ

Sắp theo visitor → thời gian → chỉ số dòng gốc để chuẩn bị tạo phiên. `event_id` chỉ ổn định thứ tự lưu trữ; các sự kiện đồng thời vẫn là một nhóm, không có thứ tự tương tác giả.

Nếu có báo cáo làm sạch lần trước của cùng CSV (khớp SHA-256), đối chiếu số dòng và số trùng. Báo cáo được đọc trước khi ghi kết quả mới, không cần báo cáo raw audit cũ.

In [ ]:
events = events.sort_values(["visitorid", "event_time", "event_id"]).reset_index(drop=True)
require(not events[REQUIRED].isna().any().any(), "Bảng sạch còn thiếu trường bắt buộc.")
require(not events.duplicated(subset=COLUMNS).any(), "Còn trùng sau ép kiểu; cần xem lại biểu diễn ID nguồn.")
require(events["event_id"].is_unique, "event_id không duy nhất.")
require(events["event_time"].notna().all(), "Bảng sạch còn NaT.")
require(str(events["event_time"].dt.tz) == "UTC", "Thời gian chưa ở UTC.")
within_visitor_gap = events.groupby("visitorid", sort=False)["timestamp"].diff()
require(within_visitor_gap.dropna().ge(0).all(), "Thời gian trong visitor chưa tăng dần.")

if previous_report and previous_report.get("source", {}).get("sha256") == source_before["sha256"]:
    require(len(events) == previous_report["clean_rows"], "Số dòng sạch khác lần trước trên cùng CSV.")
    require(duplicate_count == previous_report["exact_duplicate_rows_removed"], "Số dòng trùng khác lần trước trên cùng CSV.")
    print("Số dòng và số trùng khớp lần làm sạch trước của cùng đầu vào.")

display(events.head())
display(events.dtypes.astype(str).rename("output_dtype"))

## 7. Lưu Parquet và đọc lại để kiểm tra

Parquet giữ kiểu dữ liệu, giá trị thiếu và timezone cho các bước tiếp theo. Ghi vào tệp tạm, đọc lại và so sánh toàn bộ bảng trước khi thay thế tệp kết quả chính. Đây là đầu ra từ bước làm sạch, chưa có session hoặc nhãn train/validation/test.

In [ ]:
require(fingerprint(SOURCE) == source_before, "Raw thay đổi trong lúc chạy; cần chạy lại từ đầu.")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
temporary_output = OUTPUT.with_name(OUTPUT.name + ".tmp")
try:
    events.to_parquet(temporary_output, engine="pyarrow", index=False)
    reloaded = pd.read_parquet(temporary_output, engine="pyarrow")
    pd.testing.assert_frame_equal(events, reloaded, check_exact=True)
    temporary_output.replace(OUTPUT)
    del reloaded
finally:
    temporary_output.unlink(missing_ok=True)

print("Đã lưu và đọc kiểm tra:", OUTPUT.relative_to(ROOT).as_posix())
print(f"Số dòng: {len(events):,}")
print(f"Dung lượng: {OUTPUT.stat().st_size / 1024**2:.2f} MiB")

## 8. Lưu báo cáo làm sạch

Báo cáo ghi phiên bản Python/thư viện, SHA-256 đầu vào/đầu ra, các quy tắc và số dòng trước/sau. Chỉ ghi báo cáo thành công sau khi đọc lại Parquet và xác nhận raw giữ nguyên. Nếu chương trình dừng trước cell cuối, báo cáo thành công cũ có thể vẫn còn; xem `generated_at_utc` và mã băm để phân biệt.

In [ ]:
raw_unchanged = fingerprint(SOURCE) == source_before
require(raw_unchanged, "Raw thay đổi trước khi hoàn tất báo cáo.")
report = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "python_version": sys.version.split()[0],
    "pyarrow_version": pa.__version__,
    "notebook": "notebooks/01_kiem_tra_va_lam_sach_events.ipynb",
    "pandas_version": pd.__version__,
    "source": {"path": SOURCE.relative_to(ROOT).as_posix(), **source_before},
    "raw_rows": len(raw),
    "schema_violations": {str(key): int(value) for key, value in flags.sum().items()},
    "exact_duplicate_rows_removed": duplicate_count,
    "duplicates_by_event": duplicates_by_event,
    "clean_rows": len(events),
    "clean_event_counts": {
        str(key): int(value) for key, value in events["event"].value_counts().items()
    },
    "time_min_utc": events["event_time"].min().isoformat(),
    "time_max_utc": events["event_time"].max().isoformat(),
    "output_dtypes": {key: str(value) for key, value in events.dtypes.items()},
    "rules": {
        "deduplicate_on": COLUMNS,
        "integer_min": 0,
        "integer_max": 9223372036854775807,
        "keep_repeated_interactions_at_different_times": True,
        "timestamp_unit": "ms",
        "timezone": "UTC",
        "expected_time_start_inclusive": expected_start.isoformat(),
        "expected_time_end_exclusive": expected_end_exclusive.isoformat(),
        "sort_by": ["visitorid", "event_time", "event_id"],
    },
    "raw_unchanged_during_processing": raw_unchanged,
    "parquet_roundtrip_verified": True,
    "output": {
        "path": OUTPUT.relative_to(ROOT).as_posix(), **fingerprint(OUTPUT),
        "content_sha256": content_sha256(events, ["event_id"]), "content_key": ["event_id"],
    },
}
report_path = REPORT_PATH
write_json(report_path, report)
print("Báo cáo:", report_path.relative_to(ROOT).as_posix())
display(pd.Series({
    "raw_rows": len(raw),
    "duplicates_removed": duplicate_count,
    "clean_rows": len(events),
    "invalid_rows": int(bad_rows.sum()),
}, name="summary"))

## Sau notebook này

Đầu ra cần có:

- `data/processed/events_clean.parquet`: dữ liệu sự kiện sạch, có `event_id` và `event_time` UTC.
- `reports/events_clean/metrics.json`: kết quả thực tế của lần xử lý.

Với raw hiện tại, dự kiến 2.756.101 dòng đầu vào, loại 460 dòng trùng và giữ 2.755.641 dòng. Các số được tính trực tiếp; chỉ đối chiếu báo cáo khi cùng mã băm đầu vào.

Tiếp tục notebook 02 để tạo phiên, nhóm timestamp và chia tập. Lịch sử `categoryid`/`available`, cây danh mục và bộ mẫu prefix–target sẽ được chuẩn bị sau; không dùng snapshot cuối kỳ để điền cho quá khứ.